## Date presentation

In [ ]:
%pip install pandas
%pip install statsmodels

In [2]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import bds
from statsmodels.stats.diagnostic import linear_reset
from statsmodels.stats.diagnostic import het_arch
from statsmodels.formula.api import ols
from scipy import stats
from scipy.stats import f_oneway
from statsmodels.multivariate.manova import MANOVA

In [27]:
df = pd.read_csv("data.csv")

In [28]:
mete_col = ['Diurnal temperature range (°C)', 'Mean temperature (°C)', 'Mean humidity (%)', 'Solar radiation (MJ/m²)', 'Air pressure (hPa)', 'Dew point temperature (°C)', 'Wind speed (km/h)', 'Mean UV']
flu_col = ["H1", "H3", "B", "AandB"]


In [15]:
tdf = df[~(((df["From"] > "2003-02-21") & (df["From"] < "2003-06-23")) | ((df["From"] > "2020-01-23") & (df["From"] < "2023-03-01")) | (df["From"] < "2004-01-01"))]


---
Feature Engineering

In [ ]:

feature_df = df.copy()
feature_df

In [ ]:
feature_df = df

In [ ]:
temp_df_feature = df_feature[["From", "AandB", "Diurnal temperature range (°C)"]]
temp_df_feature

In [ ]:
temp_df_feature["AandB_Lag_1"] = temp_df_feature["AandB"].shift(1)
temp_df_feature["Temp_Lag_1"] = temp_df_feature["Diurnal temperature range (°C)"].shift(1)
temp_df_feature["AandB_Rolling_Mean_4"] = temp_df_feature["AandB"].shift(1).rolling(4).mean()
temp_df_feature["Temp_Rolling_Mean_4"] = temp_df_feature["Diurnal temperature range (°C)"].shift(1).rolling(4).mean()
temp_df_feature["_Lag_1"] = temp_df_feature["AandB_Lag_1"] / temp_df_feature["Temp_Lag_1"]
temp_df_feature["Season"] = ["Summer" if temp_df["Week"] < 0 elif ]

temp_df_feature

# col_Lag_NoOfWeek

---

ANOVA / MANOVA
- cannot use, as requires linearity

In [24]:
model = ols("Week ~ H1+H3+B", data=df).fit()


anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                 sum_sq     df          F        PR(>F)
H1         10759.353295    1.0  54.694563  4.007393e-13
H3           151.916203    1.0   0.772257  3.798190e-01
B           7360.322890    1.0  37.415784  1.581698e-09
Residual  138685.523585  705.0        NaN           NaN


In [ ]:
print("ANOVA table based on Ordinary Least Square (OLS) regression: ")
for col in mete_col:
    print(f"\n{col}: ")
    model = ols(f"{col} ~ H1 + H3 + B + AandB", data=df).fit()
    
    anova_table = sm.stats.anova_lm(model, typ=2)
    print(anova_table)

In [32]:
manova_model = MANOVA(endog=df[mete_col], exog=df["Week"])

print(manova_model.mv_test())


                  Multivariate linear model
                                                              
--------------------------------------------------------------
           x0           Value  Num DF  Den DF  F Value  Pr > F
--------------------------------------------------------------
          Wilks' lambda 0.1469 8.0000 701.0000 508.6824 0.0000
         Pillai's trace 0.8531 8.0000 701.0000 508.6824 0.0000
 Hotelling-Lawley trace 5.8052 8.0000 701.0000 508.6824 0.0000
    Roy's greatest root 5.8052 8.0000 701.0000 508.6824 0.0000



---
Test for linearity


As all the results says that the relation are non-linear, we should avoid using linear models, instead use non-linear model

In [ ]:
for val in df.columns[3:]:
    stat = df[val].describe()
    stat = stat[["mean", "std"]]
    stat["var"] = stat["std"]**2
    stat.drop(index="std", inplace=True)
    print(stat)
    print("")

In [ ]:
df = pd.read_csv("data.csv")

mete_col = ['Diurnal temperature range (°C)', 'Mean temperature (°C)', 'Mean humidity (%)', 'Solar radiation (MJ/m²)', 'Air pressure (hPa)', 'Dew point temperature (°C)', 'Wind speed (km/h)', 'Mean UV']
flu_col = ["H1", "H3", "B", "AandB"]
for i in mete_col:
    for j in flu_col:
        print(f"\nFor the columns {i} and {j}: ")
        
        # --- Fit a standard linear regression model ---
        X_with_const = sm.add_constant(df[i])
        linear_model = sm.OLS(df[j], X_with_const).fit()

        # ---- TEST 1: Ramsey RESET Test ----
        reset_test = linear_reset(linear_model, power=3, test_type='fitted')
        print("--- Ramsey RESET Test ---")
        print(reset_test.pvalue)

        if reset_test.pvalue < 0.05:
            print("Conclusion       : Non linear⚠️")
        else:
            print("Conclusion       : Linear ✅")
            
        # ---- TEST 2: BDS Test ----
        residuals = linear_model.resid
        bds_stat, bds_pvals = bds(residuals, max_dim=3)
        print("--- BDS Test (on residuals) ---")
        print(bds_pvals)
        
        
        def p_value_from_ci(lower, upper, confidence=0.95):
            alpha = 1 - confidence
            z_crit = stats.norm.ppf(1 - alpha / 2)
            estimate = (upper + lower) / 2
            se = (upper - lower) / (2 * z_crit)
            
            # Handle edge case where interval width is zero
            if se == 0:
                return 0.0 if estimate != 0 else 1.0
                
            # 4. Calculate the absolute Z-statistic
            z_stat = abs(estimate / se)
            
            # 5. Calculate the two-tailed p-value
            # stats.norm.sf(z) calculates 1 - Phi(z) directly and is more stable
            p_val = 2 * stats.norm.sf(z_stat)
            
            return p_val
        
        # --- Example Usage ---
        # Suppose your 95% Confidence Interval is [0.5, 2.5]
        lower_limit = 0.5
        upper_limit = 2.5
        
        p = p_value_from_ci(lower_limit, upper_limit, confidence=0.95)
        
        print("--- Derived Statistics ---")
        print(f"Derived p-value  : {p:.6f}")
        
        if p < 0.05:
            print("Conclusion       : Non linear⚠️")
        else:
            print("Conclusion       : Linear ✅")
        
        
 
        lm_stat, lm_p_val, f_stat, f_p_val = het_arch(residuals)
        print("--- ARCH-LM test ---")
        print(f"ARCH-LM p-value: {lm_p_val:.5f}")
        if p < 0.05:
            print("Conclusion       : Non linear⚠️")
        else:
            print("Conclusion       : Linear ✅")
        
